<a href="https://colab.research.google.com/github/night-marionette/Pnuemonia-Triage/blob/main/01_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
OUT_DIR = '/content/drive/MyDrive/PneumoniaTriage'
os.makedirs(OUT_DIR, exist_ok=True)
print("Folder ready:", OUT_DIR)

Folder ready: /content/drive/MyDrive/PneumoniaTriage


In [3]:
!pip install medmnist -q
from medmnist import PneumoniaMNIST

train_ds = PneumoniaMNIST(split="train", download=True)
val_ds   = PneumoniaMNIST(split="val",   download=True)
test_ds  = PneumoniaMNIST(split="test",  download=True)

print("Train:", len(train_ds), "| Val:", len(val_ds), "| Test:", len(test_ds))

100%|██████████| 4.17M/4.17M [00:06<00:00, 644kB/s]


Train: 4708 | Val: 524 | Test: 624


In [7]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# A simple neural network for images
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 2) # 2 classes: Normal or Pneumonia
        )
    def forward(self, x):
        return self.net(x)

# Set up GPU and model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

print("Training started...")
for epoch in range(5):
    model.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs = imgs.float().to(device)
        labels = labels.squeeze().long().to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/5 | Loss: {total_loss/len(train_loader):.3f}")

print("Training complete!")

Training started...
Epoch 1/5 | Loss: 0.479
Epoch 2/5 | Loss: 0.236
Epoch 3/5 | Loss: 0.171
Epoch 4/5 | Loss: 0.150
Epoch 5/5 | Loss: 0.144
Training complete!


In [8]:
import pandas as pd
import numpy as np

# 1. Load the test data (no shuffle! We need the original order)
test_ds = PneumoniaMNIST(split="test", download=True, transform=transform)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

model.eval()
all_scores = []
all_labels = []

# 2. Get the scores
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.float().to(device)
        outputs = model(imgs)
        # Get the probability that the image is Pneumonia (class 1)
        probs = torch.softmax(outputs, dim=1)[:, 1]

        all_scores.extend(probs.cpu().numpy())
        all_labels.extend(labels.squeeze().numpy())

all_labels = np.array(all_labels)
all_scores = np.array(all_scores)
preds = (all_scores >= 0.5).astype(int) # 0.5 is our cutoff

# 3. Calculate Baseline vs Model (Judges love this)
base_preds = np.ones_like(all_labels) # Baseline: always guess pneumonia
print("=== BASELINE (Always guess Pneumonia) ===")
print(f"Accuracy: {(base_preds == all_labels).mean()*100:.1f}%")
print(f"Normal recall (healthy kids caught): {(base_preds[all_labels==0] == 0).mean()*100:.1f}%")

print("\n=== OUR MODEL ===")
print(f"Accuracy: {(preds == all_labels).mean()*100:.1f}%")

# Calculate Recall and False Alarms
tp = ((preds==1) & (all_labels==1)).sum()
fn = ((preds==0) & (all_labels==1)).sum()
fp = ((preds==1) & (all_labels==0)).sum()

print(f"Pneumonia recall (sick kids caught): {tp / (tp + fn)*100:.1f}% | Missed (False Negatives): {fn}")
print(f"False alarms (healthy kids flagged): {fp}")

# 4. Save to Drive for Golova
df = pd.DataFrame({
    "id": range(len(all_labels)),
    "true_label": all_labels,
    "score": all_scores
})
save_path = '/content/drive/MyDrive/PneumoniaTriage/test_scores.csv'
df.to_csv(save_path, index=False)
print(f"\nSaved! File is ready for Golova at: {save_path}")

=== BASELINE (Always guess Pneumonia) ===
Accuracy: 62.5%
Normal recall (healthy kids caught): 0.0%

=== OUR MODEL ===
Accuracy: 83.8%
Pneumonia recall (sick kids caught): 97.9% | Missed (False Negatives): 8
False alarms (healthy kids flagged): 93

Saved! File is ready for Golova at: /content/drive/MyDrive/PneumoniaTriage/test_scores.csv
